# Pupil extraction

This notebook runs the pupil pipeline in three stages: strict clock/count validation, a sparse session-wide segmentation QC montage, and the full streaming extraction. The full pass is intentionally not started if the decoded video count and `cameraFrameSync` pulse count disagree. Camera timing comes directly from the shared-clock sync HDF5; no separate frametimes CSV is needed.

In [ ]:
from pathlib import Path
from dataclasses import replace
import importlib, os, sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse

# Works when the notebook is opened from either the repository root or analysis/.
REPO = Path.cwd().resolve()
if not (REPO / 'analysis').is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import analysis.session.pupil as pupil_module
pupil_module = importlib.reload(pupil_module)

from analysis.session.pupil import (
    PupilConfig, count_csv_frames, count_video_frames,
    extract_pupil, fit_pupil_mask, iter_gray_frames, order_pupil_videos,
    stage_pupil_videos, stage_sync_file, launch_pupil_tuner, load_pupil_tuning,
    segment_bright_pupil, validate_alignment_counts,
)
from analysis.session.respiration import find_behavior_sync
from analysis.session.sync import frame_onset_samples, open_sync
from analysis.session.devshim import LocalGroup
from analysis.session.resolve import resolve_group


## Configuration

Set `GROUP_ID`. The group supplies acquisition IDs, odor timing, and states; no segmentation or trace round is required. Under the experiment, the resolver expects exactly two pupil recordings somewhere below `video/`, each with a `converted/` directory containing one MP4. The earlier recording is pre and the later recording is post. `ROI` is `(y0, y1, x0, x1)` and should tightly contain the eye.

In [ ]:
# ---- what to run -------------------------------------------------------
GROUP_ID = 217
MANIPULATION = 'ketxyl'
APPROVED_ONLY = False
STAGE_VIDEO_LOCALLY = True  # strongly recommended for SMB/network storage
STAGE_SYNC_LOCALLY = True   # avoids scanning cameraFrameSync over SMB

SCRATCH = Path(os.environ.get('ODYN_SCRATCH_ROOT', str(Path.home() / 'odyn_scratch')))
SCRATCH.mkdir(parents=True, exist_ok=True)
MAIN = Path(os.environ.get('ODYN_IMAGING_ROOT', '/Volumes/MossLab/ImagingData'))
LIVE_DB = MAIN / '.odyn' / 'odyn.db'
SNAPSHOT = SCRATCH / 'odyn_snapshot.db'

group = LocalGroup(LIVE_DB, MAIN, snapshot_to=SNAPSHOT, refresh=True)
session = resolve_group(
    group, group_id=GROUP_ID, manipulation=MANIPULATION,
    approved_only=APPROVED_ONLY,
)

OUT_DIR = session.output_dir / 'aux'

# Discover <experiment>/video/<pre-or-post-pupil>/converted/<movie>.
video_roots = [p for name in ('video', 'videos') if (p := session.exp_dir / name).is_dir()]
video_roots += sorted(p for p in session.exp_dir.glob('*_video') if p.is_dir())
video_roots = list(dict.fromkeys(video_roots))
if not video_roots:
    raise FileNotFoundError(
        f'No video/, videos/, or <experiment>_video directory under {session.exp_dir}'
    )
converted_dirs = sorted({p for root in video_roots for p in root.rglob('converted') if p.is_dir()})
discovered_mp4s = []
for converted in converted_dirs:
    mp4s = sorted(converted.glob('*.mp4'))
    if len(mp4s) != 1:
        continue
    discovered_mp4s.append(mp4s[0])

if len(discovered_mp4s) != 2:
    found = [str(p.relative_to(session.exp_dir)) for p in converted_dirs]
    raise FileNotFoundError(
        'Expected exactly two pupil folders, each containing converted/<one mp4>. '
        f'Found MP4s: {[str(p) for p in discovered_mp4s]}; converted folders: {found}'
    )
SERVER_VIDEO_PATHS, VIDEO_STAMPS = order_pupil_videos(discovered_mp4s)
VIDEO_STAGE_DIR = SCRATCH / 'pupil_video_cache' / f'group{GROUP_ID}'
VIDEO_PATHS = (stage_pupil_videos(SERVER_VIDEO_PATHS, VIDEO_STAGE_DIR)
               if STAGE_VIDEO_LOCALLY else SERVER_VIDEO_PATHS)
SYNC_PATH = None  # auto-discovered from the group experiment directory

print(f'group:       {GROUP_ID} ({session.exp_name})')
for state, server_path, path, (stamp, source) in zip(
    ('pre', 'post'), SERVER_VIDEO_PATHS, VIDEO_PATHS, VIDEO_STAMPS
):
    print(f'{state:5s} source: {server_path.relative_to(session.exp_dir)}')
    print(f'      decode: {path}')
    print(f'             {stamp.isoformat()} ({source})')
print(f'output:      {OUT_DIR.relative_to(session.exp_dir)}')

ROI = (100, 300, 80, 280)  # group 217 starting point: (y0, y1, x0, x1)
CONFIG = PupilConfig(
    roi=ROI,
    bright_percentile=97.0,  # raises Otsu into the bright pupil tail
    ransac_residual_px=2.0,
    ransac_trials=200,
    min_inlier_fraction=0.55,
    max_residual_px=3.0,
    min_axis_ratio=0.25,
    max_diameter_rate_px_s=150.0,
    max_bad_fraction=0.20,
)

N_QC_FRAMES = 16
PUPIL_WORKERS = 6       # exact same fits, distributed across CPU cores
PUPIL_BATCH_SIZE = 128  # bounds queued image memory
PUPIL_CHECKPOINT_EVERY = 1000
PUPIL_CHECKPOINT_DIR = SCRATCH / 'pupil_checkpoints' / f'group{GROUP_ID}'

## 1. Alignment preflight

This performs one streaming decode solely to obtain the exact decoded frame count. A mismatch raises immediately, before segmentation or output writing.

In [ ]:
for path in VIDEO_PATHS:
    if not path.exists():
        raise FileNotFoundError(path)

server_sync_path = (Path(SYNC_PATH) if SYNC_PATH is not None
                    else find_behavior_sync(session.exp_dir))
sync_path = (stage_sync_file(server_sync_path, SCRATCH / 'pupil_sync_cache' / f'group{GROUP_ID}')
             if STAGE_SYNC_LOCALLY else server_sync_path)
print(f'sync source: {server_sync_path}')
print(f'sync read:   {sync_path}')
sync = open_sync(sync_path)
camera_samples_all = frame_onset_samples(sync, channel='cameraFrameSync')
from analysis.session.sync import group_frames_into_acquisitions
camera_blocks = group_frames_into_acquisitions(camera_samples_all, rate_hz=sync.rate_hz)
if len(camera_blocks) != 2:
    raise ValueError(f'Expected two contiguous cameraFrameSync blocks, found {len(camera_blocks)}')
video_counts = [validate_alignment_counts(path, block) for path, block in zip(VIDEO_PATHS, camera_blocks)]
n_camera = sum(video_counts)
camera_samples = np.concatenate(camera_blocks)
two_p_samples = frame_onset_samples(sync, channel='2pFrameSync')
two_p_blocks = group_frames_into_acquisitions(two_p_samples, rate_hz=sync.rate_hz)
imaging_active = np.zeros(len(camera_samples), dtype=bool)
for block in two_p_blocks:
    lo = np.searchsorted(camera_samples, block[0], side='left')
    hi = np.searchsorted(camera_samples, block[-1], side='right')
    imaging_active[lo:hi] = True
camera_time_s = camera_samples / sync.rate_hz
duration_min = (camera_time_s[-1] - camera_time_s[0]) / 60

print(f'PASS pre:  {video_counts[0]:,} decoded frames = cameraFrameSync block 1 pulses')
print(f'PASS post: {video_counts[1]:,} decoded frames = cameraFrameSync block 2 pulses')
print(f'Sync: {sync_path}')
print(f'Camera duration: {duration_min:.2f} min')
print(f'Median camera rate: {1 / np.median(np.diff(camera_time_s)):.3f} Hz')
print(f'Frames inside 2p acquisitions: {imaging_active.mean():.1%}')

## 2. Tune ROI and threshold

The videos are streamed once to retain evenly spaced raw illuminated examples. The dimmest and brightest examples then open in the tuner. Set the four ROI corner sliders (`x0`, `x1`, `y0`, `y1`); the yellow rectangle is mirrored on both panels. Adjust the adaptive-threshold offset until cyan segmentation is correct on both, then save.

In [ ]:
per_video_qc = max(2, N_QC_FRAMES // 2)
raw_samples = []
offset = 0
for state, path, count in zip(('pre', 'post'), VIDEO_PATHS, video_counts):
    local_active = np.flatnonzero(imaging_active[offset:offset + count])
    if local_active.size == 0:
        raise ValueError(f'No illuminated camera frames overlap the {state} video')
    sample_indices = set(local_active[np.linspace(0, local_active.size - 1, per_video_qc, dtype=int)])
    for local_i, frame in enumerate(iter_gray_frames(path)):
        if local_i not in sample_indices:
            continue
        raw_samples.append((state, local_i, frame))
    offset += count

y0, y1, x0, x1 = CONFIG.roi or (0, raw_samples[0][2].shape[0], 0, raw_samples[0][2].shape[1])
brightness = np.array([np.mean(item[2][y0:y1, x0:x1]) for item in raw_samples])
dim_frame = raw_samples[int(np.argmin(brightness))][2]
bright_frame = raw_samples[int(np.argmax(brightness))][2]
TUNING_PATH = OUT_DIR / f'group{GROUP_ID}_{session.exp_name}_pupil_tuning.json'
tuner = launch_pupil_tuner(
    dim_frame, bright_frame, save_path=TUNING_PATH, roi=CONFIG.roi,
    threshold_offset=CONFIG.threshold_offset,
    bright_percentile=CONFIG.bright_percentile,
)

After saving in the GUI, apply the settings below.

In [ ]:
if not TUNING_PATH.exists():
    raise FileNotFoundError('Click Save ROI + threshold in the tuner first.')
tuning = load_pupil_tuning(TUNING_PATH)
CONFIG = replace(CONFIG, **tuning)
print('Applied:', tuning)

## 3. Sparse segmentation QC

The saved ROI and threshold are now applied to all retained raw examples. Cyan is the segmentation, magenta is the chord-excluded RANSAC ellipse, and the montage is saved before full extraction.

In [ ]:
samples = []
for state, local_i, frame in raw_samples:
    mask, threshold = segment_bright_pupil(
        frame, roi=CONFIG.roi, bright_percentile=CONFIG.bright_percentile,
        threshold_offset=CONFIG.threshold_offset,
    )
    cleaned, fit = fit_pupil_mask(
        mask, CONFIG, random_seed=CONFIG.random_seed + local_i,
    )
    samples.append((state, local_i, frame, mask, cleaned, threshold, fit))

n_col = 4
n_row = int(np.ceil(len(samples) / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(14, 3.5 * n_row), squeeze=False)
for ax, item in zip(axes.flat, samples):
    state, i, frame, mask, cleaned, threshold, fit = item
    ax.imshow(frame, cmap='gray')
    excluded = mask & ~cleaned
    if np.any(excluded):
        yellow = np.zeros((*excluded.shape, 4), dtype=float)
        yellow[excluded] = (1.0, 0.85, 0.0, 0.75)
        ax.imshow(yellow)
    ax.contour(cleaned, levels=[0.5], colors='cyan', linewidths=0.8)
    if fit is not None:
        ax.add_patch(Ellipse(
            (fit['x'], fit['y']), 2 * fit['major'], 2 * fit['minor'],
            angle=np.degrees(fit['theta']), fill=False,
            edgecolor='magenta', linewidth=1.2,
        ))
        detail = (f"in={fit['inlier_fraction']:.2f}  chord={fit.get('chord_fraction', np.nan):.2f}  "
                  f"tail={fit.get('concavity_fraction', 0.0):.2f}  "
                  f"res={fit['residual']:.2f}  d={fit['diameter']:.1f}")
    else:
        detail = 'FIT FAILED'
    ax.set_title(f'{state} frame {i:,}  t={threshold:.0f}\n{detail}', fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(samples):]:
    ax.axis('off')
fig.suptitle('Sparse pupil QC: yellow excluded tail, cyan retained mask, magenta ellipse')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_QC_PATH = OUT_DIR / f'group{GROUP_ID}_{session.exp_name}_pupil_preflight_qc.png'
fig.savefig(INTERMEDIATE_QC_PATH, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {INTERMEDIATE_QC_PATH}')

Inspect the montage now. If the cyan component is not consistently the pupil, tighten `ROI`. If thresholds drift onto sclera or background, correct the ROI or illumination before proceeding. Re-run the configuration and sparse-QC cells after changes.

## 4. Full streaming extraction

This is the long pass. It repeats the strict count preflight by design, processes one frame at a time, aligns camera frame `i` to camera pulse `i`, and samples onto the 2p frame grid using the shared clock.

In [ ]:
if 'TUNING_PATH' in globals() and TUNING_PATH.exists():
    CONFIG = replace(CONFIG, **load_pupil_tuning(TUNING_PATH))
    print('Using saved pupil tuning:', load_pupil_tuning(TUNING_PATH))

report = extract_pupil(
    VIDEO_PATHS, sync_path,
    acq_ids=session.acq_ids, odor_ids=session.odor_ids, states=session.states,
    odor_on_frames=session.odor_on_frames, odor_off_frames=session.odor_off_frames,
    frame_rate=session.frame_rate, exp_name=f'group{GROUP_ID}_{session.exp_name}',
    trial_ids=session.table['trial_id'].to_numpy(),
    out_dir=OUT_DIR, config=CONFIG, save=True,
    validate_counts=False, validated_counts=video_counts,
    workers=PUPIL_WORKERS, batch_size=PUPIL_BATCH_SIZE,
    checkpoint_every=PUPIL_CHECKPOINT_EVERY,
    checkpoint_dir=PUPIL_CHECKPOINT_DIR, resume=True,
)

print(f"HDF5: {report['h5']}")
print(f"QC figure: {report['figure']}")
print(f"Trials: {report['n_trial']}, frames/trial: {report['n_frame']}")
print(f"Flagged trials: {int(report['flagged'].sum())}/{report['n_trial']}")
print(f"Median blink fraction: {np.median(report['blink_fraction']):.2%}")
print(f"Median clipping fraction: {np.median(report['clipped_fraction']):.2%}")

## 5. Final QC outputs

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(report['figure'])))

worst_trials = np.argsort(report['masked_fraction'])[::-1][:10]
print('Worst trials by masked fraction:')
for row in worst_trials:
    print(
        f"  trial row {row:3d}, acq_id={int(report['acq_id'][row])}, "
        f"masked={report['masked_fraction'][row]:.1%}, "
        f"blink={report['blink_fraction'][row]:.1%}, "
        f"clipped={report['clipped_fraction'][row]:.1%}"
    )